# AI-Based Tornado Risk Prediction and Early Warning Research System
### *Using Historical Weather and Wind Data in the Indian Subcontinent*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Python](https://img.shields.io/badge/Python-3.9%20%7C%203.10%20%7C%203.11-blue.svg)](https://www.python.org/)
[![Target Domain: India](https://img.shields.io/badge/Target%20Region-India%20(Pre--monsoon)-red.svg)](#)

---

> ### ⚠️ Critical Educational & Research Prototype Disclaimer
> **This notebook is an academic research prototype developed strictly for educational and scientific demonstration purposes.**
>
> - **NOT an Operational Warning System:** This notebook is **NOT** an operational early-warning or nowcasting tool and MUST NOT be used for public safety advisories, disaster management, or life-safety decisions.
> - **Official Warning Authority in India:** In the Republic of India, official severe weather advisories, thunderstorm nowcasts, squall alerts, and extreme weather bulletins are issued solely by the **India Meteorological Department (IMD)**, under the **Ministry of Earth Sciences (MoES)**.
> - **Scientific Reality:** Severe weather conditions (e.g., high wind shear, narrow dew point depression) indicate severe convective *potential*, but they do **NOT** prove that a tornado touched down.
> - **No Fabricated Labels:** All historical tornado records analyzed in this notebook come from verified IMD monographs and peer-reviewed literature (e.g., Peterson & Mehta, 1981).
> - **No Black-Box Modeling Before Verification:** We inspect data quality, atmospheric physics, and class balance before conducting supervised modeling.

## Section 1: Environment & Dynamic Path Setup

In this cell, we ensure that the required libraries are installed and that Python can find our project files whether running locally on your computer or in a Google Colab virtual machine.

In [ ]:
import sys
import os

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running inside Google Colab environment.')
    # Install required dependencies
    !pip install -q pandas numpy matplotlib seaborn requests
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('src', exist_ok=True)
else:
    print('Running in local Python environment.')

# Ensure project root is in Python sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root directory set to: {PROJECT_ROOT}')

## Section 2: Library Imports & Visual Styling

Here we import standard scientific Python packages:
- **`pandas` (`pd`)**: For structured tabular data manipulation (DataFrames).
- **`numpy` (`np`)**: For numerical computations and trigonometric vector math.
- **`matplotlib.pyplot` (`plt`)**: For foundational plotting and chart rendering.
- **`seaborn` (`sns`)**: For enhanced statistical data visualization.
- **`requests`**: For querying the ECMWF ERA5 reanalysis weather API.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Configure visual styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')
print('Libraries imported and visual styling initialized.')

## Section 3: Loading the Verified Indian Tornado Catalog

In this section, we load the verified historical catalog of documented Indian tornadoes.
These events were assembled from published India Meteorological Department (IMD) post-disaster surveys, *MAUSAM* journal articles, and the landmark climatological study by **Peterson & Mehta (1981)**.

No tornado occurrences have been fabricated.

In [ ]:
catalog_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'indian_tornado_catalog.csv')

if os.path.exists(catalog_path):
    df_catalog = pd.read_csv(catalog_path)
else:
    # Standalone Colab fallback: try importing from src or fetch directly
    try:
        from src.data_loader import load_indian_tornado_catalog
        df_catalog = load_indian_tornado_catalog(catalog_path)
    except Exception:
        # Fallback to loading directly from the GitHub repository raw URL
        raw_url = 'https://raw.githubusercontent.com/Prince/ai-based-tornado-risk-prediction/main/data/raw/indian_tornado_catalog.csv'
        df_catalog = pd.read_csv(raw_url)

df_catalog['date'] = pd.to_datetime(df_catalog['date'])
print(f'Successfully loaded {len(df_catalog)} verified historical Indian tornado events.')
df_catalog[['event_id', 'date', 'time_ist', 'district_or_place', 'state', 'estimated_intensity', 'source_citation']].head(10)

## Section 4: Geographical Distribution of Indian Tornado Events

Let us visualize the spatial distribution of historical Indian tornadoes.
Notice how the events form two distinct geographical clusters:
1. **The Eastern / Northeastern Corridor:** West Bengal, Odisha, Assam, and Jharkhand (active during pre-monsoon *Kalbaishakhi* / Nor'westers from March to May).
2. **The Northwestern Corridor:** Punjab, Haryana, and Delhi (triggered by mid-latitude Western Disturbances or pre-monsoon dust-squall lines).

In [ ]:
plt.figure(figsize=(10, 7))
scatter = sns.scatterplot(
    data=df_catalog,
    x='longitude',
    y='latitude',
    hue='state',
    style='estimated_intensity',
    s=140,
    alpha=0.9
)

# Annotate notable benchmark events
for _, row in df_catalog.iterrows():
    if any(k in str(row['district_or_place']) for k in ['Delhi', 'Kendrapara', 'Jalpaiguri']):
        short_name = str(row['district_or_place']).split('(')[0].strip()
        plt.annotate(
            f"{short_name} ({row['date'].year})",
            xy=(row['longitude'], row['latitude']),
            xytext=(row['longitude'] + 0.4, row['latitude'] + 0.3),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
            fontweight='bold',
            fontsize=9
        )

plt.title('Geographical Distribution of Documented Indian Tornado Events', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Longitude (°E)', fontsize=11)
plt.ylabel('Latitude (°N)', fontsize=11)
plt.xlim(70, 95)
plt.ylim(18, 33)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Section 5: Atmospheric Reanalysis Ingestion (ERA5 via Open-Meteo)

Now we query the **ECMWF ERA5 reanalysis archive** using the Open-Meteo API.
We focus on a verified severe pre-monsoon storm period in Eastern India:
- **Location:** Jalpaiguri / Mainaguri, West Bengal ($26.56^\circ\text{ N}, 88.82^\circ\text{ E}$).
- **Date Range:** March 30 to April 1, 2024 (encompassing the severe pre-monsoon storm and mini-tornado of March 31, 2024, ~15:30 IST).
- **Variables:** Surface temperature, dew point, surface pressure, 10m wind speed, 100m wind speed, and 10m wind gusts.

In [ ]:
from src.data_loader import fetch_era5_hourly

# Jalpaiguri event coordinates
LAT, LON = 26.5626, 88.8204
START_DATE = '2024-03-30'
END_DATE = '2024-04-01'

print(f'Fetching ERA5 reanalysis for ({LAT}°N, {LON}°E) from {START_DATE} to {END_DATE}...')
df_era5 = fetch_era5_hourly(
    latitude=LAT,
    longitude=LON,
    start_date=START_DATE,
    end_date=END_DATE,
    timezone='Asia/Kolkata'
)

print(f'Data received: {len(df_era5)} hourly records.')
df_era5.head(5)

## Section 6: Data Cleaning, Quality & Missing Value Audit

In professional machine learning, never feed data into a model without validating its quality.
Here we:
1. Audit for missing (null) values.
2. Check physical range plausibility (temperatures between -10°C and 55°C, positive wind speeds, valid pressures).
3. Confirm that the hourly time-series is strictly continuous.

In [ ]:
from src.data_loader import validate_atmospheric_dataframe

# 1. Check null counts
null_report = df_era5.isnull().sum()
print('Null values per column:')
print(null_report)

# 2. Run automated physical plausibility audit
audit_results = validate_atmospheric_dataframe(df_era5)
print('\nValidation Audit Status:', audit_results['status'])
print(f'Total rows: {audit_results["total_rows"]}, Total nulls: {audit_results["total_nulls"]}')

# 3. Statistical summary
df_era5[['temperature_2m', 'dew_point_2m', 'surface_pressure', 'wind_speed_10m', 'wind_gusts_10m']].describe()

## Section 7: Meteorological Feature Engineering & Derivatives

Ordinary surface temperature alone cannot predict severe convective phenomena. We must compute scientifically justified atmospheric derivatives:
1. **Dew Point Depression ($T_{\text{diff}} = T - T_d$):** Low values indicate that the air is near saturation, leading to low Lifting Condensation Levels (LCL) favorable for severe storms.
2. **Low-Level Speed Shear ($\Delta V_{100-10} = V_{100\text{m}} - V_{10\text{m}}$):** Measures vertical wind speed shear in the lowest 100 meters.
3. **Hourly Pressure Tendency ($\Delta P / \Delta t$):** Measures the rate of surface pressure drops preceding storm fronts.
4. **Gust Factor ($V_{\text{gust}} / V_{\text{sustained}}$):** Measures boundary-layer turbulence.

In [ ]:
from src.data_loader import compute_atmospheric_derivatives

# Compute physical features
df_features = compute_atmospheric_derivatives(df_era5)

print(f'Augmented dataset shape: {df_features.shape}')
print('New derived meteorological features:')
df_features[['dew_point_depression', 'low_level_speed_shear', 'pressure_tendency_1h', 'gust_factor']].head(5)

## Section 8: Exploratory Weather & Wind Data Visualizations

Now we plot the thermodynamic and kinematic conditions over the 72-hour period.
Notice:
- The sharp daytime heating and subsequent pressure drops.
- The vertical wind speed shear between 10m and 100m.
- The spike in wind gusts during the afternoon convective window on March 31, 2024.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# 1. Thermodynamic plot: Temperature vs Dew Point
axes[0].plot(df_features.index, df_features['temperature_2m'], label='2m Temperature (°C)', color='#d95f02', lw=2)
axes[0].plot(df_features.index, df_features['dew_point_2m'], label='2m Dew Point (°C)', color='#1b9e77', lw=2, linestyle='--')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('1. Thermodynamic Profile: Temperature vs. Dew Point (Jalpaiguri Corridor)', fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# 2. Kinematic plot: 10m vs 100m Wind Speed (Vertical Shear)
axes[1].plot(df_features.index, df_features['wind_speed_10m'], label='10m Wind Speed (km/h)', color='#7570b3', lw=1.8)
axes[1].plot(df_features.index, df_features['wind_speed_100m'], label='100m Wind Speed (km/h)', color='#e7298a', lw=1.8)
axes[1].fill_between(df_features.index, df_features['wind_speed_10m'], df_features['wind_speed_100m'], color='#e7298a', alpha=0.15, label='Vertical Speed Shear')
axes[1].set_ylabel('Wind Speed (km/h)')
axes[1].set_title('2. Kinematic Profile: 10m vs. 100m Winds (Vertical Speed Shear)', fontweight='bold')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

# 3. Peak Wind Gusts & Pressure Tendency
ax3_pressure = axes[2].twinx()
axes[2].bar(df_features.index, df_features['wind_gusts_10m'], width=0.03, color='#e6ab02', alpha=0.7, label='Peak 10m Gusts (km/h)')
ax3_pressure.plot(df_features.index, df_features['surface_pressure'], color='#66a61e', lw=2, label='Surface Pressure (hPa)')
axes[2].set_ylabel('Wind Gusts (km/h)', color='#d95f02')
ax3_pressure.set_ylabel('Surface Pressure (hPa)', color='#66a61e')
axes[2].set_title('3. Storm Dynamics: Surface Pressure Drops vs. Peak Wind Gusts', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Highlight approximate event hour (March 31, 2024 ~15:30 IST)
event_time = pd.Timestamp('2024-03-31 15:00:00')
for ax in axes:
    ax.axvline(event_time, color='red', linestyle=':', lw=2, alpha=0.8)

plt.tight_layout()
plt.show()

## Section 9: Preventing Data Leakage with Chronological Splitting

In standard tabular machine learning (like predicting house prices), researchers randomly shuffle rows using `train_test_split(shuffle=True)`.

**In meteorology and atmospheric science, random shuffling causes severe data leakage!**

Why? Because atmospheric observations are strongly autocorrelated across time. If you train on 2:00 PM and test on 3:00 PM of the same storm, the model simply memorizes the existing storm rather than learning true predictive patterns.

To prevent leakage, all weather models must use **chronological (time-series) train/test splits**.

In [ ]:
# Demonstrate chronological time-series splitting
split_cutoff = pd.Timestamp('2024-03-31 12:00:00')

train_df = df_features[df_features.index < split_cutoff]
test_df = df_features[df_features.index >= split_cutoff]

print('Strict Chronological Train/Test Split (Zero Data Leakage):')
print(f'Training Period: {train_df.index.min()} to {train_df.index.max()} ({len(train_df)} hours)')
print(f'Testing Period:  {test_df.index.min()} to {test_df.index.max()} ({len(test_df)} hours)')

# Assert no overlap
assert train_df.index.max() < test_df.index.min(), 'Leakage Error: Training set overlaps with testing period!'
print('\n[VERIFIED] Temporal separation strictly enforced. Zero future information leaks into training data.')

## Section 10: Notebook Self-Test & Stage 3 Verification

This final cell runs automated unit tests inside the notebook to confirm that all data ingestion, feature calculations, and splits executed successfully.

In [ ]:
# Automated Self-Test Cell
print('=' * 65)
print('RUNNING STAGE 3 NOTEBOOK SELF-TESTS...')
print('=' * 65)

# 1. Verify Catalog
assert len(df_catalog) == 13, f'Expected 13 verified events, found {len(df_catalog)}'
assert any('Jalpaiguri' in str(v) for v in df_catalog['district_or_place'].values), 'Jalpaiguri event missing!'
print('[PASS] 1. Historical Indian catalog integrity verified.')

# 2. Verify Atmospheric DataFrame
assert len(df_features) == 72, f'Expected 72 hourly records, found {len(df_features)}'
assert df_features.isnull().sum().sum() == 0, 'Unexpected null values in features!'
print('[PASS] 2. ERA5 hourly time-series continuity and data completeness verified.')

# 3. Verify Derived Meteorological Features
required_features = ['dew_point_depression', 'low_level_speed_shear', 'pressure_tendency_1h', 'gust_factor']
for feat in required_features:
    assert feat in df_features.columns, f'Missing feature: {feat}'
    assert not df_features[feat].isnull().all(), f'Feature {feat} contains only nulls!'
print('[PASS] 3. Convective kinematic and thermodynamic feature derivations verified.')

# 4. Verify Leakage Prevention
assert train_df.index.max() < test_df.index.min(), 'Chronological split violated!'
print('[PASS] 4. Chronological train/test split verified with zero data leakage.')

# 5. Scientific Integrity Policy
assert 'tornado_occurred' not in df_features.columns, (
    'Integrity violation: Severe weather variables must not be labeled as proof of a tornado!'
)
print('[PASS] 5. Scientific integrity policy verified (no unvalidated tornado labels).')

print('\n' + '=' * 65)
print('STAGE 3 GOOGLE COLAB NOTEBOOK VERIFICATION COMPLETE: ALL CHECKS PASSED!')
print('=' * 65)